In [1]:
import os
import psycopg2

conn = psycopg2.connect(
    host='postgres',
    port=5432,
    dbname=os.environ.get('POSTGRES_DB', 'pipeline'),
    user=os.environ.get('POSTGRES_USER', 'pipeline_user'),
    password=os.environ.get('POSTGRES_PASSWORD', 'pipeline_pass')
)
cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS app.events (
        event_id   SERIAL PRIMARY KEY,
        user_id    INT NOT NULL,
        event_type VARCHAR(64) NOT NULL,
        payload    JSONB,
        created_at TIMESTAMP NOT NULL DEFAULT NOW()
    )
""")

cur.execute("""
    INSERT INTO app.events (user_id, event_type, payload) VALUES
        (1, 'click',        '{"button": "buy_now"}'::jsonb),
        (2, 'purchase',     '{"item_id": 42, "amount": 99.99}'::jsonb),
        (3, 'login',        NULL),
        (4, 'signup',       '{"referral": "promo2024"}'::jsonb),
        (5, 'logout',       NULL),
        (6, 'page_view',    '{"page": "/home"}'::jsonb),
        (7, 'add_to_cart',  '{"item_id": 17}'::jsonb),
        (8, 'checkout',     '{"total": 149.50, "currency": "USD"}'::jsonb)
""")

conn.commit()

cur.execute('SELECT count(*) FROM app.events')
row_count = cur.fetchone()[0]
print(f'Rows in app.events: {row_count}')

cur.close()
conn.close()

Rows in app.events: 8


In [3]:
import os
import psycopg2
import json
import datetime
from kafka import KafkaProducer

conn = psycopg2.connect(
    host='postgres',
    port=5432,
    dbname=os.environ.get('POSTGRES_DB', 'pipeline'),
    user=os.environ.get('POSTGRES_USER', 'pipeline_user'),
    password=os.environ.get('POSTGRES_PASSWORD', 'pipeline_pass')
)
cur = conn.cursor()

cur.execute("""
    SELECT event_id, user_id, event_type, payload, created_at
    FROM app.events
""")
rows = cur.fetchall()
cur.close()
conn.close()

producer = KafkaProducer(
    bootstrap_servers='redpanda:9092',
    value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8')
)

count = 0
for row in rows:
    event_id, user_id, event_type, payload, created_at = row
    message = {
        'event_id': event_id,
        'user_id': user_id,
        'event_type': event_type,
        'payload': json.dumps(payload) if payload is not None else None,
        'created_at': created_at.isoformat() if isinstance(created_at, datetime.datetime) else str(created_at)
    }
    producer.send('events-raw', value=message)
    count += 1

producer.flush()
print(f'Produced {count} messages to topic events-raw')

Produced 8 messages to topic events-raw


## Step 3: Run Flink Job

Two options to run the Flink streaming job that reads from Redpanda and writes to MinIO S3.

**Option A — Flink SQL Client:**

```bash
docker exec -it flink-jobmanager ./bin/sql-client.sh
```

Once the SQL client opens, paste the full contents of `flink-jobs/sql/e2e_redpanda_to_s3.sql`.

**Option B — PyFlink:**

```bash
docker exec -it flink-jobmanager flink run --python /opt/flink/jobs/python/e2e_redpanda_to_s3.py
```

**What to expect:**

The job is a streaming job. After consuming all existing messages from the `events-raw` topic, it keeps running and waits for new messages. This is expected behavior for streaming pipelines.

Wait 15-20 seconds after the job starts for the first checkpoint to complete. Then check the MinIO Console at http://localhost:9001 — navigate to the `raw` bucket and look for files under the `events/` prefix.

To stop the job after verification: open the Flink Dashboard at http://localhost:8081, find the running job, and click Cancel.

In [ ]:
import os
os.environ['AWS_REGION'] = 'us-east-1'

from pyspark.sql import SparkSession

pg_user = os.environ.get('POSTGRES_USER', 'pipeline_user')
pg_password = os.environ.get('POSTGRES_PASSWORD', 'pipeline_pass')
aws_key = os.environ.get('AWS_ACCESS_KEY_ID', 'minioadmin')
aws_secret = os.environ.get('AWS_SECRET_ACCESS_KEY', 'minioadmin')

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('iceberg-e2e-test')
    .config('spark.jars.packages',
            'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,'
            'org.apache.iceberg:iceberg-aws-bundle:1.7.1,'
            'org.apache.hadoop:hadoop-aws:3.3.4,'
            'org.postgresql:postgresql:42.7.4')
    .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.iceberg.type', 'jdbc')
    .config('spark.sql.catalog.iceberg.uri',
            'jdbc:postgresql://postgres:5432/pipeline?currentSchema=iceberg_catalog')
    .config('spark.sql.catalog.iceberg.jdbc.user', pg_user)
    .config('spark.sql.catalog.iceberg.jdbc.password', pg_password)
    .config('spark.sql.catalog.iceberg.jdbc.schema-version', 'V1')
    .config('spark.sql.catalog.iceberg.warehouse', 's3a://warehouse/')
    .config('spark.sql.catalog.iceberg.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.iceberg.s3.endpoint', 'http://minio:9000')
    .config('spark.sql.catalog.iceberg.s3.path-style-access', 'true')
    .config('spark.sql.catalog.iceberg.s3.region', 'us-east-1')
    .config('spark.sql.catalog.iceberg.client.region', 'us-east-1')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', aws_key)
    .config('spark.hadoop.fs.s3a.secret.key', aws_secret)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .getOrCreate()
)

spark.sql('CREATE NAMESPACE IF NOT EXISTS iceberg.db')

df = spark.read.json('s3a://raw/events/')
df.printSchema()
print(f'Row count from S3: {df.count()}')

df.writeTo('iceberg.db.events').createOrReplace()

spark.sql("""
    SELECT count(*) as cnt, event_type
    FROM iceberg.db.events
    GROUP BY event_type
    ORDER BY event_type
""").show(20, False)

spark.stop()